# Fine-tuning BLIP — descrição de condição do livro

Antes de rodar:

1. **Add Input** (barra lateral) → anexar o Kaggle Dataset das fotos:
   `pedrocanoas/fotos-livros`. O código vem direto do GitHub (célula 1),
   não precisa mais de um segundo Dataset.
2. **Settings** → Accelerator: **GPU (T4 x2 ou P100)** · Internet: **On**
   (necessário tanto pro `git clone` quanto pra baixar os pesos do BLIP do
   Hugging Face na primeira execução)

> **Nota sobre o layout do Kaggle:** o Kaggle monta os datasets em
> `/kaggle/input/datasets/<usuário>/<slug>/...` (não mais direto em
> `/kaggle/input/<slug>/...`). Confirme rodando `!find /kaggle/input -maxdepth 4`
> se os caminhos abaixo não baterem com a estrutura do seu notebook.

In [ ]:
# Célula 1 — clona o código direto do GitHub (sempre pega a versão mais recente do main)
!git clone --depth 1 https://github.com/Pedrocanoas/tcc.git /kaggle/working/repo
%cd /kaggle/working/repo/model

In [ ]:
# Célula 2 — instala as deps que faltam (não reinstala torch: o Kaggle já vem com build CUDA pronta)
!pip install -q -r requirements-kaggle.txt
# o Kaggle traz um torchao antigo (0.10); o peft (LoRA do BLIP-2) aborta ao encontrar versão
# abaixo da 0.16, mesmo sem usar — nada aqui depende de torchao, então sai
!pip uninstall -y -q torchao

In [ ]:
# Célula 3 — aponta os diretórios de dados/checkpoints para /kaggle/working (única área gravável)
import os

os.environ["TCC_DATASET_RAW_DIR"] = "/kaggle/input/datasets/pedrocanoas/fotos-livros"
os.environ["TCC_DATASET_PROCESSED_DIR"] = "/kaggle/working/processed"
os.environ["TCC_CHECKPOINTS_DIR"] = "/kaggle/working/checkpoints"

In [ ]:
# Célula 4 — rodada da comparação justa: todos os modelos com a MESMA base limpa
# (códigos internos removidos, coleções fora do treino via --exclude-colecoes), o MESMO
# split de validação (192 livros) e as MESMAS métricas (src.metrics).
# --max-length 128: cobre ~91-92% das legendas inteiras nos dois tokenizadores.
#
# Esta execução roda só o BLIP-2 (OPT-2.7b): Q-Former + LoRA no modelo de linguagem,
# treino e avaliação no mesmo script (o modelo é grande demais pra baixar e avaliar
# localmente). 10 epochs com o melhor checkpoint escolhido pela validação: com 5, a
# perda de validação ainda estava caindo na última. Os BLIP já foram treinados e
# avaliados com esta mesma base — comandos e resultados em model/README.md:
#   kaggle_train.py --epochs 20 --batch-size 8                      (384, encoder congelado)
#   kaggle_train.py ... --image-size 576                            (576, encoder congelado)
#   kaggle_train.py ... --unfreeze-vision-layers 2 --vision-lr 1e-5 (384 e 576 destravados;
#                       o 576 com --batch-size 4 --grad-accum 2 pra caber na GPU)
!python -m src.data.prepare_dataset --exclude-colecoes
!python -u -m src.blip2_experiment --epochs 10 --batch-size 4 --output-dir /kaggle/working/ckpt_blip2 > /kaggle/working/eval_blip2.log 2>&1; grep -E "versões|treináveis|== |Error" /kaggle/working/eval_blip2.log; tail -5 /kaggle/working/eval_blip2.log

In [ ]:
# Célula 5 (opcional) — sanity check: gera uma legenda de exemplo com o checkpoint treinado
import glob

sample_photos = sorted(glob.glob("/kaggle/input/datasets/pedrocanoas/fotos-livros/livro001/foto*.jpg"))[:1]
if sample_photos:
    !python -m src.infer {sample_photos[0]}
else:
    print("ajuste o caminho de exemplo acima para uma foto existente no seu dataset")

In [ ]:
# Célula 6 — remove os checkpoints "last" (só o "best" interessa) pra encurtar o download
# do Output; as pastas best/ já aparecem no Output sem precisar de zip
!rm -rf /kaggle/working/ckpt_*/last
!du -sh /kaggle/working/ckpt_*/best

Depois de rodar tudo, clique em **Save Version** (Save & Run All) para commitar o
notebook — as pastas `ckpt_*/best/` aparecem na aba **Output** para baixar direto
pelo navegador, ou via Kaggle API:

```sh
kaggle kernels output <seu-usuario>/<slug-do-notebook> -p ./kaggle-output
```

Copie a pasta `best/` escolhida para `model/checkpoints/best/` (caminho que
`infer.py` usa por padrão localmente).